# Task 3: Category articles with BART

Use the five meta-categories exported by the weighted clustering notebook. This experiment uses `sshleifer/distilbart-cnn-12-6`, a 306M-parameter distilled BART summarizer, smaller than the suggested 1B?8B range and suitable for a CPU baseline. It is pretrained for news summarization, so review-domain quality must be inspected. No proprietary API or fine-tuning is used.

Select a notebook kernel with the project dependencies from `requirements.txt` installed. The first generation downloads the checkpoint (about 1.2 GB). Source reviews stay local. Articles and sampled evidence are saved in `experiments/bart_category_articles/`.

Model card: https://huggingface.co/sshleifer/distilbart-cnn-12-6


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'experiments' / 'summarize_categories_bart.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'experiments' / 'summarize_categories_bart.py').exists(), 'Run from repository root or experiments directory.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from experiments.summarize_categories_bart import prepare_data, product_stats, select_evidence, BartSummarizer, generate_articles
DATA_PATH = ROOT / 'data/reviews_with_meta_categories.csv'
df = prepare_data(DATA_PATH)
display(df.groupby('meta_category').agg(review_rows=('_row', 'size'), profiles=('_profile', 'nunique')))


## Evidence and ranking

We keep the `(id, name, brand, categories)` profiles from Task 2 because IDs and names are inconsistent. Top three profiles have the highest mean rating among profiles with at least 10 rated reviews. The lowest-rated eligible profile is a caution section, not automatic advice to avoid a product. Statistics use all source rows; invalid or missing ratings are excluded from rating calculations.

For each section, select up to eight distinct review texts, cycling through profiles and prioritizing helpful votes. Separate 4?5-star praise from 1?2-star complaints. These are bounded samples, not an exhaustive summary or a measure of complaint prevalence. BART processes all tokens of selected evidence in chunks below its context limit. Chunk summaries are joined to preserve minority points.


In [ ]:
for category, rows in df.groupby('meta_category'):
    print(category)
    display(product_stats(rows).sort_values('mean_rating', ascending=False).head(5))


## Generate all five articles

BART produces the prose; Python supplies headings and verified rating statistics. It is not an instruction-following chat model, so detailed blog-writing prompts are not assumed to work. Generation uses deterministic beam search. Cached section summaries avoid repeated inference. CPU generation can take several minutes.


In [ ]:
summarizer = BartSummarizer(variant='plain')
OUTPUT_DIR = ROOT / 'experiments/bart_category_articles'
manifest = generate_articles(df, summarizer, OUTPUT_DIR, min_reviews=10)
assert len(manifest) == df['meta_category'].nunique()
manifest


In [ ]:
from IPython.display import Markdown, display
for item in manifest:
    display(Markdown((OUTPUT_DIR / item['article']).read_text(encoding='utf-8')))


## Compare source-framing variants

Run the same negative-review evidence through (A) raw review text and (B) a short descriptive introductory sentence. Both use identical decoding. This is a source-framing experiment rather than a claim that BART follows instructions. Compare faithfulness, complaint coverage, readability, and unsupported claims manually before choosing a variant. No reference articles exist, so ROUGE would not establish factual quality.


In [ ]:
category = 'Cases and protective covers'
selected = select_evidence(df[df['meta_category'].eq(category) & df['_rating'].le(2)])
display(selected[['_row', '_profile', '_rating', '_text']])
texts = selected['_text'].tolist()
plain = summarizer.summarize(texts, 'complaints')
summarizer.variant = 'framed'
try:
    framed = summarizer.summarize(texts, 'complaints')
finally:
    summarizer.variant = 'plain'
display(Markdown(f'**A ? Raw reviews**\n\n{plain}\n\n**B ? Framed source**\n\n{framed}'))


## Manual evaluation and next steps

For each article, inspect its evidence JSON and record: unsupported claims (target zero), coverage of selected complaints, factual product attribution, repetition, and readability (1?5). Compare strengths and complaints across the top products without inventing specifications. Check especially names that conflict with review content. Rankings may be sensitive to duplicate rows and small samples.

If output remains poor after source-framing and decoding experiments, collect human-written review summaries, split by product/profile to prevent leakage, then fine-tune BART and evaluate on held-out products. This notebook establishes an inference baseline; it does not fabricate training targets or claim measured accuracy.


## Results from the local CPU experiment

All five category articles were generated. Open `bart_category_articles/index.md` for articles and evidence, `quality_review.md` for observed failures, and `source_framing_comparison.json` for the two tested variants.

The framing experiment changes detail selection without clearly improving factuality. The streaming article invents a John Sutter attribution absent from sampled evidence. Missing and conflicting product names also prevent reliable named-product recommendations. These outputs establish a working BART baseline, not publication-ready buying advice. Verify identities and facts before publishing, and consider review-domain fine-tuning with human summaries if quality remains inadequate.
